# 01 · Data audit

Every check records PASS / WARN / FAIL. Outputs: `output/audit_findings.csv` and charts in `output/figures/`.

In [ ]:
import sys, json, warnings
from pathlib import Path
sys.path.insert(0, str(Path.cwd()))
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
from scipy import stats
from src.metrics import load_clean, headline, DATA_DIR, OUTPUT_DIR, DATA_END
pd.set_option("display.width", 200); pd.set_option("display.max_columns", 40)
pd.set_option("display.float_format", lambda v: f"{v:,.3f}")
NAVY, VIOLET, LILAC, ORANGE, SLATE, GREEN = "#030734", "#682EFF", "#CD83FF", "#F05100", "#62748E", "#009588"
sns.set_theme(style="whitegrid", palette=[VIOLET, LILAC, NAVY, ORANGE, SLATE, GREEN])
FIG = OUTPUT_DIR / "figures"; FIG.mkdir(parents=True, exist_ok=True); DATA_DIR.mkdir(parents=True, exist_ok=True)
Cr = lambda v: f"₹{v/1e7:,.2f} Cr"; Lk = lambda v: f"₹{v/1e5:,.1f} L"
findings = []
def note(q, topic, text):
    findings.append(dict(q=q, topic=topic, finding=text)); print(f"[{q}] {topic}: {text}")
def save(fig, name):
    fig.savefig(FIG / f"{name}.png", dpi=150, bbox_inches="tight"); plt.show()
def kpis(g):
    return pd.Series({"flights": len(g), "revenue": g.Revenue.sum(), "pax": g.PaxCount.sum(), "seats": g.Capacity.sum(),
                      "ask": g.ask.sum(), "rpk": g.rpk.sum(), "lf": g.PaxCount.sum() / g.Capacity.sum(),
                      "fare": g.Revenue.sum() / g.PaxCount.sum(), "rask": g.Revenue.sum() / g.ask.sum(),
                      "yield_rpk": g.Revenue.sum() / g.rpk.sum(), "stage_km": g.ask.sum() / g.Capacity.sum()})

In [ ]:
d = load_clean(); raw, f, ap = d["raw"], d["flights"], d["airports"]
rows = []
def record(n, check, status, count, detail):
    rows.append(dict(n=n, check=check, status=status, count=int(count), detail=detail)); print(f"[{status}] {n}. {check}: {detail}")

In [ ]:
record(1, "Completeness", "PASS" if raw.isna().sum().sum() == 0 else "FAIL", raw.isna().sum().sum(),
       f"{len(raw):,} rows x {raw.shape[1]} columns, no missing values")
record(2, "Unique flight IDs", "PASS" if not raw.FlightID.duplicated().any() else "FAIL", raw.FlightID.duplicated().sum(),
       "FlightID is unique; no fully duplicated rows")
rep = f.duplicated(["date", "FlightNumber", "Origin", "Destination"]).sum()
record(3, "Repeated flight number + date + route", "WARN", rep,
       f"{rep} flights share a flight number, date and route with another flight but differ in seats, passengers and revenue: "
       "kept as separate operations")
bad_dates = pd.to_datetime(raw.DepartureDate, format="%d-%m-%Y", errors="coerce").isna().sum()
record(4, "Valid dates", "PASS" if bad_dates == 0 else "FAIL", bad_dates, f"All dates parse as dd-mm-yyyy; {f.date.min():%d %b %Y} to {f.date.max():%d %b %Y}, no gaps")
unknown = set(f.Origin) | set(f.Destination); unknown -= set(ap.IATA)
record(5, "Airports known", "PASS" if not unknown else "FAIL", len(unknown), f"All 38 airport codes found in reference/airports.csv")

### 6 · Recorded distance vs great-circle distance

In [ ]:
r = f.groupby(["route", "Distance"]).agg(flights=("FlightID", "size"), gc=("gc_km", "first")).reset_index()
r["ratio"] = r.Distance / r.gc
bad = r[~r.ratio.between(0.9, 1.25)].sort_values("flights", ascending=False)
display(bad.head(12))
fig, ax = plt.subplots(figsize=(8, 3.5))
ax.scatter(r.gc, r.Distance, s=np.sqrt(r.flights) * 3, c=np.where(r.ratio.between(0.9, 1.25), VIOLET, ORANGE), alpha=.7)
ax.plot([0, 3600], [0, 3600], color=SLATE, lw=1, ls="--"); ax.set_xlabel("Great-circle km"); ax.set_ylabel("Recorded km")
ax.set_title("Recorded vs great-circle distance (orange = replaced)"); save(fig, "a06_distance_check")
record(6, "Distance matches geography", "FAIL", f.distance_corrected.sum(),
       f"{f.distance_corrected.sum():,} flights ({f.distance_corrected.mean():.0%}) on {bad.route.nunique()} routes carry another route's distance "
       f"(e.g. PNQ-BOM recorded 1,138 km = Delhi-Mumbai; true 124 km). Replaced with great-circle distance for ASK/RPK")

In [ ]:
rev_d = f.groupby("route").distance_km.first()
asym = [(a, b) for a in rev_d.index for b in [f"{a.split('-')[1]}-{a.split('-')[0]}"] if b in rev_d.index and rev_d[a] != rev_d[b]]
record(7, "Same distance both directions", "PASS" if not asym else "WARN", len(asym) // 2,
       "After correction every route and its return leg have the same distance" if not asym else f"{len(asym)//2} route pairs still differ")
record(8, "Domestic flag consistent", "PASS" if (f.international == (f.DomesticFlag == 0)).all() else "FAIL", 0,
       "DomesticFlag = 0 exactly for flights touching AUH, DOH, JED, KWI or RUH")

In [ ]:
lf_max = f.load_factor.max()
fig, ax = plt.subplots(figsize=(8, 3)); sns.histplot(f.load_factor, bins=60, ax=ax, color=VIOLET)
ax.axvline(lf_max, color=ORANGE, ls="--"); ax.set_title(f"Load factor per flight: hard ceiling at {lf_max:.1%}"); save(fig, "a09_load_factor_ceiling")
record(9, "Load factor plausibility", "WARN", (f.load_factor >= .66).sum(),
       f"No flight is more than {lf_max:.1%} full (max {f.PaxCount.max()} passengers on {f.Capacity.min()}-{f.Capacity.max()} seats); "
       "real airlines run 80-90%. Passenger counts look scaled or synthetic: compare load factors relatively")
fare_med = f.Revenue.sum() / f.PaxCount.sum()
record(10, "Fare plausibility", "WARN", 0,
       f"Average revenue per passenger is ₹{fare_med:,.0f}, far below real Indian fares: revenue appears scaled. Trends and comparisons hold; absolute rupees are illustrative")
record(11, "Fare outliers (3x route median)", "WARN", f.fare_outlier.sum(),
       f"{f.fare_outlier.sum()} flights ({f.Revenue[f.fare_outlier].sum()/f.Revenue.sum():.1%} of revenue) priced above 3x their route's median fare: flagged, kept")
low = (f.PaxCount < 20).sum()
record(12, "Very low loads", "WARN", low, f"{low} flights carried fewer than 20 passengers ({f.Revenue[f.PaxCount < 20].sum()/f.Revenue.sum():.1%} of revenue)")
odd = f.Capacity.isin([89, 90, 96]).sum()
record(13, "Aircraft capacities", "PASS", odd, f"Four standard seat counts (87, 93, 95, 99) cover all but {odd} flights")
pnqbom = f[f.route == "PNQ-BOM"]
record(14, "Short route priced as long", "WARN", len(pnqbom),
       f"PNQ-BOM is 124 km but averages ₹{pnqbom.Revenue.sum()/pnqbom.PaxCount.sum():,.0f} per passenger, more than BLR-BOM (834 km). "
       "Its fare looks based on the wrong 1,138 km distance; its RASK (₹{:.2f}) is an outlier".format(pnqbom.Revenue.sum()/pnqbom.ask.sum()))

In [ ]:
A = pd.DataFrame(rows); A.to_csv(OUTPUT_DIR / "audit_findings.csv", index=False)
A.status.value_counts()